In [0]:
# Databricks notebook source
# =====================================================
# 1️⃣ Define Connection Variables (EDIT THESE)
# =====================================================

sqlserver_host = "ashishsqlservr.database.windows.net"
sqlserver_port = "1433"
sqlserver_database = "ashishDatabase"
sqlserver_user = "ashishsqlservr"
sqlserver_password = "Ashish@123"

# Secret scope name (will be created if not exists)
secret_scope_name = "banking-scope"

# Secret key name (single secret containing full JSON)
secret_key_name = "sqlserver-connection-json"
# secret_key_name = "sqlserver-connection-json-dummy"

# COMMAND ----------

In [0]:
# =====================================================
# 2️⃣ Build JSON Object
# =====================================================

import json

connection_config = {
    "host": sqlserver_host,
    "port": sqlserver_port,
    "database": sqlserver_database,
    "user": sqlserver_user,
    "password": sqlserver_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

connection_json = json.dumps(connection_config)

print("Generated JSON Configuration:")
print(connection_json)


# COMMAND ----------

# Python cell in the same workspace notebook
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

api_url  = ctx.apiUrl().getOrElse(None)     # e.g. https://adb-...azuredatabricks.net
api_token = ctx.apiToken().getOrElse(None)  # personal access token for this session

print(api_url)
print(api_token)  # handle securely, do not log in real code


import requests
import json

# ----------------------------------------
# Configuration
# ----------------------------------------
DATABRICKS_INSTANCE = api_url  # Replace with your workspace URL
DATABRICKS_TOKEN = api_token  # Replace with your PAT

scope_name = secret_scope_name  # Scope to be created
backend_type = "DATABRICKS"     # Use "AZURE_KEYVAULT" if integrating with Key Vault

# COMMAND ----------



# ----------------------------------------
# API Endpoint
# ----------------------------------------
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/create"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope_name
}

# ----------------------------------------
# Send request
# ----------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

# ----------------------------------------
# Handle response
# ----------------------------------------
if response.status_code == 200:
    print(f"Secret scope '{scope_name}' created successfully.")
else:
    print("Failed to create secret scope.")
    print("Status Code:", response.status_code)
    print("Response:", response.text)


# COMMAND ----------

import requests
import json

scope = secret_scope_name          # Already existing scope
secret_key = secret_key_name       # Name of the secret entry
secret_value = connection_json # Value to store securely

# -------------------------------------------------
# API Endpoint
# -------------------------------------------------
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope,
    "key": secret_key,
    "string_value": secret_value
}

# -------------------------------------------------
# Send Request
# -------------------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

# -------------------------------------------------
# Output
# -------------------------------------------------
if response.status_code == 200:
    print(f"Secret '{secret_key}' created successfully in scope '{scope}'.")
else:
    print("Failed to create secret.")
    print("Status:", response.status_code)
    print("Response:", response.text)


# COMMAND ----------

# =====================================================
# 5️⃣ Verify Secret Retrieval
# =====================================================

try:
    retrieved_json = dbutils.secrets.get(
        scope=secret_scope_name,
        key=secret_key_name
    )
    
    print("Secret retrieved successfully.")
    
    parsed = json.loads(retrieved_json)
    print("Parsed JSON:")
    print(parsed)
    
except Exception as e:
    print("Secret verification failed:")
    print(str(e))


# COMMAND ----------

import requests
import json

scope = secret_scope_name          # Already existing scope
secret_key = 'gmail_api_key'       # Name of the secret entry
secret_value = '<Your gmail api key>' # Value to store securely

# -------------------------------------------------
# API Endpoint
# -------------------------------------------------
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope,
    "key": secret_key,
    "string_value": secret_value
}

# -------------------------------------------------
# Send Request
# -------------------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

# -------------------------------------------------
# Output
# -------------------------------------------------
if response.status_code == 200:
    print(f"Secret '{secret_key}' created successfully in scope '{scope}'.")
else:
    print("Failed to create secret.")
    print("Status:", response.status_code)
    print("Response:", response.text)

In [0]:
dbutils.secrets.listScopes() 

In [0]:
import json

config = json.loads(
    dbutils.secrets.get(
        scope="banking-scope",
        key="sqlserver-connection-json"
    )
)

jdbc_url = (
    f"jdbc:sqlserver://{config['host']}:{config['port']};"
    f"databaseName={config['database']};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "loginTimeout=30;"
)

test_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "(SELECT 1 AS connection_test) AS test")
    .option("user", config["user"])
    .option("password", config["password"])
    .option("driver", config["driver"])
    .load()
)

test_df.show()

In [0]:
%sh
nc -zv ashishsqlservr.database.windows.net 1433

In [0]:
import json
import traceback

config = json.loads(
    dbutils.secrets.get(
        scope="banking-scope",
        key="sqlserver-connection-json"
    )
)

jdbc_url = (
    f"jdbc:sqlserver://{config['host']}:{config['port']};"
    f"databaseName={config['database']};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "loginTimeout=30;"
)

try:
    result = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", "(SELECT 1 AS test) AS src")
        .option("user", config["user"])
        .option("password", config["password"])
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .load()
    )

    result.show()
    print("JDBC connection successful")

except Exception as e:
    print("Exception type:", type(e).__name__)
    print("Exception details:")
    traceback.print_exception(type(e), e, e.__traceback__)

In [0]:
import json
import traceback

config = json.loads(
    dbutils.secrets.get(
        scope="banking-scope",
        key="sqlserver-connection-json"
    )
)

jdbc_url = (
    "jdbc:sqlserver://ashishsqlservr.database.windows.net:1433;"
    "databaseName=ashishDatabase;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "loginTimeout=30;"
)

try:
    df = (
        spark.read.format("jdbc")
        .option("url", jdbc_url)
        .option("query", "SELECT 1 AS connection_test")
        .option("user", config["user"])
        .option("password", config["password"])
        .option("driver", config["driver"])
        .load()
    )

    df.show()
    print("JDBC test successful")

except Exception as e:
    print("ERROR TYPE:", type(e).__name__)
    print("ERROR MESSAGE:", str(e)[:3000])

    current = e
    for i in range(8):
        cause = getattr(current, "__cause__", None)
        if cause is None:
            cause = getattr(current, "__context__", None)
        if cause is None:
            break

        print(f"\nCAUSE {i + 1}: {type(cause).__name__}")
        print(str(cause)[:2000])
        current = cause